# Auditoria dos CSVs de trafego (ANTT)

Primeira olhada nos dados brutos: schema, encoding, tipos.
Fase F1/F2 do plano de tarefas.

In [1]:
import pandas as pd
from pathlib import Path

print('pandas', pd.__version__)

pandas 3.0.6


In [7]:
ARQ = Path("../data/raw/volume_pedagio_2024.csv")

df = pd.read_csv(ARQ, sep=';', encoding='latin-1', decimal=',', nrows=100);

print(df.shape)
print(df.dtypes)

df.head()

(100, 8)
concessionaria         str
mes_ano                str
sentido                str
praca                  str
tipo_cobranca          str
categoria_eixo       int64
tipo_de_veiculo        str
volume_total       float64
dtype: object


,concessionaria,mes_ano,sentido,praca,tipo_cobranca,categoria_eixo,tipo_de_veiculo,volume_total
0,AUTOPISTA FERNÃO DIAS,01/01/2024,Crescente,1 Sul (Mairiporã),Automática,10,Comercial,13.0
1,AUTOPISTA FERNÃO DIAS,01/01/2024,Crescente,1 Sul (Mairiporã),Automática,11,Comercial,8.0
2,AUTOPISTA FERNÃO DIAS,01/01/2024,Crescente,1 Sul (Mairiporã),Automática,12,Comercial,6.0
3,AUTOPISTA FERNÃO DIAS,01/01/2024,Crescente,1 Sul (Mairiporã),Automática,2,Comercial,235.0
4,AUTOPISTA FERNÃO DIAS,01/01/2024,Crescente,1 Sul (Mairiporã),Automática,2,Passeio,20802.0


In [ ]:
for ano in [2024, 2025, 2026]:
    arq = Path(f"../data/raw/volume_pedagio_{ano}.csv")
    df = pd.read_csv(arq, sep=';', decimal = ',', encoding = 'latin-1', nrows=100_000);
    print("Ano:", ano, " | Shape:", df.shape)
    print(df.dtypes)
    print("-" * 40)

In [12]:
# os conjuntos de valores de sentido, tipo_cobranca e tipo_de_veiculo são iguais nos 3 anos?

from pathlib import Path
import pandas as pd

COLS = ["sentido", "tipo_cobranca", "tipo_de_veiculo"]

for ano in [2024, 2025, 2026]:
    arq = Path(f"../data/raw/volume_pedagio_{ano}.csv")
    df = pd.read_csv(arq, sep=";", encoding="latin-1", usecols=COLS)
    print("=== Ano", ano, "| linhas:", len(df))
    for c in COLS:
        print("  ", c, "->", sorted(df[c].unique()))


=== Ano 2024 | linhas: 3312763
   sentido -> ['Crescente', 'Decrescente']
   tipo_cobranca -> ['Automática', 'Manual', 'OCR/PLACA']
   tipo_de_veiculo -> ['Comercial', 'Moto', 'Passeio', 'Veículo Pequeno']
=== Ano 2025 | linhas: 3274157
   sentido -> ['Crescente', 'Decrescente']
   tipo_cobranca -> ['Automática', 'Manual', 'OCR/PLACA']
   tipo_de_veiculo -> ['Comercial', 'Moto', 'Passeio']
=== Ano 2026 | linhas: 2597234
   sentido -> ['Crescente', 'Decrescente']
   tipo_cobranca -> ['Automática', 'Manual', 'OCR/PLACA']
   tipo_de_veiculo -> ['Comercial', 'Moto', 'Passeio']


In [13]:
# como foi encontrado um tipo de veículo único em 2024 que não estava nos outros anos, vamos verificar se o valor ou quantidade de ocorrências desse tipo de veículo é relevante para o modelo. 
# Se for irrelevante, podemos simplesmente removê-lo do dataset.

ARQ = Path("../data/raw/volume_pedagio_2024.csv")

df = pd.read_csv(
    ARQ,
    sep=";",
    encoding="latin-1",
    decimal=",",
    usecols=["tipo_de_veiculo", "volume_total"], 
)

mascara = df["tipo_de_veiculo"] == "Veículo Pequeno"
print("linhas de Veiculo Pequeno:", mascara.sum())

vol_pequeno = df.loc[mascara, "volume_total"].sum()
vol_total = df["volume_total"].sum()

print("Veiculo Pequeno:", vol_pequeno)
print("Total 2024 .....:", vol_total)
print("Participacao ...:", vol_pequeno / vol_total)




linhas de Veiculo Pequeno: 7
Veiculo Pequeno: 7.0
Total 2024 .....: 834732408.0
Participacao ...: 8.385920964506268e-09


In [ ]:
# unificando arquivos de 2024, 2025 e 2026 em um único dataset

from pathlib import Path
import pandas as pd

frames = []
for ano in [2024, 2025, 2026]:
    arq = Path(f"../data/raw/volume_pedagio_{ano}.csv")
    df = pd.read_csv(arq, sep=";", encoding="latin-1", decimal=",")
    df["ano_origem"] = ano
    frames.append(df)
    print("lido", ano, "->", df.shape)

dados = pd.concat(frames, ignore_index=True)
print("TOTAL:", dados.shape)

lido 2024 -> (3312763, 9)


C:\Users\danfl\AppData\Local\Temp\ipykernel_21824\4093368789.py:9: DtypeWarning: Columns (5: categoria_eixo) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(arq, sep=";", encoding="latin-1", decimal=",")


lido 2025 -> (3274157, 9)
lido 2026 -> (2597234, 9)
TOTAL: (9184154, 9)
